### Postal Service Accessibility Analysis in San Diego
- Geocode post office addresses in San Diego
- Generate 5-minute and 10-minute drive-time service areas for each postal office
- Find underserved areas in San Diego that are outside every 10-minute service area
- GeoEnrich underserved areas

#### Overture Maps licensing
- Places:  [CDLA Permissive v 2.0](https://cdla.dev/permissive-2-0/)

In [0]:
import geoanalytics

In [0]:
from geoanalytics.sql import functions as ST 
from pyspark.sql import functions as F

### Read in Overture map data
- get all postal office addresses in San Diego CA

In [0]:
# load Overture map places dataset and get all postal office address in San Diego
s3_places_parquet = r"s3a://overturemaps-us-west-2/release/2026-05-20.0/theme=places/type=place/"
places = spark.read.format("parquet").load(s3_places_parquet)
post_office = places.filter("categories.primary like '%post_office%'") \
                        .withColumn("address", F.explode("addresses")) \
                        .select(
                            "id",
                            F.col("categories.primary").alias("categories_primary"),
                            F.col("names.primary").alias("names_primary"),
                            F.col("address.freeform").alias("address_freeform"),
                            F.col("address.locality").alias("address_locality"),
                            F.col("address.postcode").alias("address_postcode"),
                            F.col("address.region").alias("address_region"),
                            F.col("address.country").alias("address_country")) \
                        .where("address_country == 'US'") \
                        .where("address_region == 'CA'") \
                        .where("address_locality == 'San Diego'").persist()

### Geocode Post Office Addresses

In [0]:
# load SMP license
geoanalytics.add_data_licenses(license_file="dbfs:/FileStore/licensing/licenseKeys_us.txt")

In [0]:
# confirm SMP license has been initialized
spark.sql("select * from geoanalytics.debug.mapssdk").show(truncate=False)

In [0]:
# add California mmpk file to Spark
sc.addFile("s3a://{}/network_datasets/California.mmpk".format(s3_bucket_name))

In [0]:
from geoanalytics.tools import Geocode
# Geocode postal office addresses in San Diego
geocode_result = Geocode() \
                .setLocator("California.mmpk") \
                .setOutFields("minimal") \
                .setAddressFields("address_freeform", "address_locality", "address_postcode", "address_region") \
                .run(post_office)

In [0]:
geocode_result.printSchema()

In [0]:
# filter geocode result
post_office_sd = geocode_result \
                  .where("Score > 90") \
                  .where("Addr_type == 'PointAddress'") \
                  .select("id", "names_primary", "geocode_location").distinct().persist()

In [0]:
# plot postal office locations on a basemap
post_office_sd.st.plot(basemap="streets", geometry="geocode_location", marker_size=5, color="black")

### Create Drive-Time Service Area

In [0]:
from geoanalytics.tools import CreateServiceAreas
# create 5 and 10 mins drive time service area from each post office location
post_office_drivetime = CreateServiceAreas()\
  .setNetwork("California.mmpk")\
  .setTravelMode("DrivingTime")\
  .setCutoffs([5,10], "minutes")\
  .run(post_office_sd).persist()

In [0]:
import matplotlib.pyplot as plt

# plot a subset of the service area polygons on a basemap
colors = ['#67a9cf','#016c59']
cmap = plt.cm.colors.ListedColormap(colors)

result_plot = post_office_drivetime.filter(F.col("id").isin(ids)).st.plot(cmap_values="cutoff",
                             cmap=cmap, is_categorical=True,
                             basemap="light",
                             figsize=(7,7),
                             legend=True,
                             legend_kwds = {"title": "Cutoffs"},
                             show_axis=True);

post_office_sd.filter(F.col("id").isin(ids)).st.plot(facecolor = "#bd0026", marker_size=5, ax=result_plot, show_axis=True)

### Identify Underserved Area
- Find Census Block groups within UnderserverAreas 

In [0]:
from geoanalytics.tools import Overlay

# 10 minutes drive time service area
postal_office_service_area = post_office_drivetime.where("cutoff == '10 minutes'") \
                .withColumn("id", F.lit("1")) \
                .groupBy("id")\
                .agg(ST.aggr_union("service_area_polygon").alias("aggr_service_area"))

# get block group polygons in San Diego City
sd_city = spark.read.format("shapefile").load("abfss://{0}@{1}.dfs.core.windows.net/cities".format(container,storage_account)) \
            .where("NAME == 'San Diego'")

sd_block_group = spark.read.format("shapefile") \
                          .load("abfss://{0}@{1}.dfs.core.windows.net/us_census_block_group".format(container,storage_account)) \
                          .groupBy("ID", "NAME", "COUNTY", "STATE_NAME", "OBJECTID")\
                          .agg(ST.aggr_union("geometry").alias("bg_geometry")) \
                          .where("COUNTY == 'San Diego'")

sd_block_group_city = Clip().run(input_dataframe=sd_block_group,clip_dataframe=sd_city)

# find underserved area (block groups)
underserved_area_block_group = Overlay() \
                                .setOverlayType(overlay_type="Erase") \
                                .run(input_dataframe=sd_block_group_city, overlay_dataframe=postal_office_service_area).drop("geometry").persist()

### GeoEnrich underserved area
- Which underserved areas have the largest population and greatest need for postal services? 

In [0]:
from geoanalytics.tools import GeoEnrichVariables

# explore GeoEnrich variables that are related to population
geoenrich_dataset_path = "/databricks/geoenrich-dataset/GeoEnrich_Essentials_for_ArcGIS_GeoAnalytics_Engine_2_1_0.biz"

geoenrich_vars = GeoEnrichVariables() \
            .setDataPath(geoenrich_dataset_path) \
            .run()
geoenrich_vars.where("alias LIKE '%Population%'").show(truncate=False)

In [0]:
from geoanalytics.tools import GeoEnrich

# GeoEnrich underserved area with the selected variables
variable_list = ["*TOTPOP*", "*AGGHINC"]

geoenrich_result = GeoEnrich() \
                    .setDataPath(geoenrich_dataset_path) \
                    .setVariables(*variable_list) \
                    .run(underserved_area_block_group).persist()

In [0]:
geoenrich_result.printSchema()

### Visualize GeoEnrich result

In [0]:
# plot the underserved area (census block groups) with GeoEnrich result (Total Population)
result_plot = geoenrich_result.where("ACSTOTPOP > 0").st.plot(cmap_values="ACSTOTPOP",
                                                    legend=True,
                                                    cmap="YlOrRd",
                                                    figsize=(12,7),
                                                    basemap="light",
                                                    show_axis=True)
result_plot.set_title("2023 Total Population (ACS 5-Yr) of Postal Service Underserved Area in San Diego city")